In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())
api_key = os.environ['GROQ_API_KEY']

In [10]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=api_key        
)



In [27]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("D:\\practice AI-ML\\rag\\docs\\devops.pdf")
docs=loader.load()


 And so part of your availability is going to be, if you do have a downtime, can you 
have your systems up and running within 20 minutes? If you can, that's going to 
minimize that downtime. It still happened, but the disruption to your end users 
has been minimized as much as possible. 
 We haven't talked about scaling yet, but having an application that can handle 
lots and lots of users is going to mitigate (ease) a popular application that's going 
to shut your application down from being too popular. And so I suggest load 
testing, and I'll tell you in a later video some stories about how load testing has 
saved my bacon (smoked meat) a few times by basically sending hundreds of 
concurrent users to your application, finding where that capacity limit is, how 
many users at once can your application handle? 
 And maybe that spurs (urge, encourage) you to fix the bottleneck to improve the 
efficiency of your application, rewrite that code, add additional servers, auto 
scaling th

str

In [40]:

from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter=RecursiveCharacterTextSplitter(
    separators=["\n\n","\n",",","."," ",""],
    chunk_size=300,
    chunk_overlap=50
    
)
chunks=splitter.split_documents(docs)
print(chunks[0].metadata)





{'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-09-22T17:42:22+05:00', 'moddate': '2025-09-22T20:23:51+05:00', 'title': 'Microsoft Word - AZ900 Section3 Video 13', 'author': '', 'source': 'D:\\practice AI-ML\\rag\\docs\\devops.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1'}


In [48]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddingModel=HuggingFaceEmbeddings(
     model_name="sentence-transformers/all-MiniLM-L6-v2"
)
texts = []
for chunk in chunks:
    texts.append(chunk.page_content)
embeddings=embeddingModel.embed_documents(texts)
print(len(embeddings[0]))


384


In [58]:
from pinecone import Pinecone
import uuid

load_dotenv(find_dotenv())

pc = Pinecone(api_key=os.environ["PineCone_API_KEY"]) 
index = pc.Index("docsembedding")
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddingModel
)

vectors = []
for chunk, embedding in zip(chunks, embeddings):
    vectors.append({
        "id": str(uuid.uuid4()),
        "values": embedding,
        "metadata": {
            "text": chunk.page_content,
            "source": chunk.metadata.get("source", "unknown"),
            "page": chunk.metadata.get("page", 0)
        }
    })


batch_size = 100
for i in range(0, len(vectors), batch_size):
    index.upsert(vectors=vectors[i : i + batch_size])
    print(f" Batch {i // batch_size + 1} uploaded")

print(index.describe_index_stats())

 Batch 1 uploaded
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 118}},
 'total_vector_count': 118,
 'vector_type': 'dense'}


In [60]:
question=input("Hello what do u want to understand from this pdf?")
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10
    }
)

docs = retriever.invoke(question)
print(docs)

Hello what do u want to understand from this pdf? unplanned outages


[Document(metadata={'page': 2.0, 'source': 'D:\\practice AI-ML\\rag\\docs\\devops.pdf'}, page_content='unplanned outages. \n\uf0d8 So, what are planned outages? It seems like an oxymoron (a figure of speech that \ncombines contradictory words with opposing meanings). But really, these are the times \nwhen you do need to make changes to your system, including security patches'), Document(metadata={'page': 3.0, 'source': 'D:\\practice AI-ML\\rag\\docs\\devops.pdf'}, page_content="perfectly fine, if you don't have access to the Internet, you can't connect to your \ncustomers, and that is unplanned outage. \n\uf0d8 You can have a power outage not only to the computer, to the rack, but to the \nwhole building. The whole data center can have a power outage."), Document(metadata={'page': 3.0, 'source': 'D:\\practice AI-ML\\rag\\docs\\devops.pdf'}, page_content='disasters like hurricanes and earthquakes and things can cause systems outage  \n      Unplanned Outages \n\uf0b7 Hardware failure \n

In [65]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful assistant.
Use the provided context to answer the user's question.
If the answer is not present in the context, say respectfully that
you could not find the answer in the provided documents.
Do not make up information.
"""
    ),
    MessagesPlaceholder(variable_name="history"),
    (
        "human",
        """
Context:
{context}
Question:
{question}
"""
    )
])

chain = (
    RunnablePassthrough.assign(
        context=lambda x: retriever.invoke(x["question"])
    )
    | prompt
    | model
)

store = {}
user = "user123"

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

response = conversational_chain.invoke(
    {"question": question},
    config={"configurable": {"session_id": user}}
)

print(response.content)

According to the provided documents, unplanned outages can be caused by:

1. Hardware failure
2. Network disruptions
3. Power Outages
4. Natural disasters (such as hurricanes and earthquakes)

Additionally, it is mentioned that unplanned outages can occur when you don't have access to the Internet, which prevents you from connecting to your customers.


In [66]:
while True:
    question = input("You: ")
    if question.lower() in ["exit", "quit"]:
        break
    
    response = conversational_chain.invoke(
        {"question": question},
        config={"configurable": {"session_id": user}}
    )
    print("Bot:", response.content)

You:  can u explain it again


Bot: According to the provided context, unplanned outages refer to situations where a system becomes unavailable or non-operational due to unexpected events. 

Examples of such events mentioned in the context include:

1. Natural disasters (such as flooding in lower Manhattan) that can affect computer systems.

The concept of high availability is related to a system's ability to remain operational for users even during planned or unplanned outages.


You:  what is ml


Bot: I couldn't find the answer to "what is ML" in the provided documents. The context appears to be related to DevOps, cloud services, and high availability, but it does not provide a definition or explanation of ML.


You:  quit


In [67]:
print(store[user])

Human: unplanned outages
AI: According to the provided documents, unplanned outages can be caused by:

1. Hardware failure
2. Network disruptions
3. Power Outages
4. Natural disasters (such as hurricanes and earthquakes)

Additionally, it is mentioned that unplanned outages can occur when you don't have access to the Internet, which prevents you from connecting to your customers.
Human: can u explain it again
AI: According to the provided context, unplanned outages refer to situations where a system becomes unavailable or non-operational due to unexpected events. 

Examples of such events mentioned in the context include:

1. Natural disasters (such as flooding in lower Manhattan) that can affect computer systems.

The concept of high availability is related to a system's ability to remain operational for users even during planned or unplanned outages.
Human: what is ml
AI: I couldn't find the answer to "what is ML" in the provided documents. The context appears to be related to DevOps